In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("train.csv")
train

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


### Exercise 1: Duplicate Detection and Removal

Objective: Identify and remove duplicate entries in the Titanic dataset.

* Load the Titanic dataset.
* Identify if there are any duplicate rows based on all columns.
* Remove any duplicate rows found in the dataset.
* Verify the removal of duplicates by checking the number of rows before and after the duplicate removal.
* Hint: Use the duplicated() and drop_duplicates() functions in Pandas.

In [ ]:
train[train.duplicated(keep=False)]

### Exercise 2: Handling Missing Values

* Identify columns in the Titanic dataset with missing values.
* Explore different strategies for handling missing data, such as removal, imputation, and filling with a constant value.
* Apply each strategy to different columns based on the nature of the data.

Hint: Review methods like dropna(), fillna(), and SimpleImputer from scikit-learn.

In [ ]:
train.isnull().sum()

In [ ]:
train_copy = train.copy()
train_copy['Embarked'] = train_copy['Embarked'].fillna(train_copy['Embarked'].mode()[0]) #only 2 values missing. Can drop but also can just add the most frequent embarkation point
train_copy['Age'] = train_copy['Age'].fillna(train_copy['Age'].median()) #too many values to drop. Instead impute the data with the median age
train_copy = train_copy.drop(columns=['Cabin'])

train_copy.isnull().sum()


### Exercise 3: Feature Engineering

* Create new features, such as Family Size from SibSp and Parch, and Title extracted from the Name column.
* Convert categorical variables into numerical form using techniques like one-hot encoding or label encoding.
* You will encode new categorical features (like Title) here, but do not scale numerical features yet — that will come after outlier handling.

Hint: Utilize Pandas for data manipulation and scikit-learn’s preprocessing module for encoding.

In [ ]:
train_copy['FamilySize'] = train_copy['SibSp'] + train_copy['Parch'] + 1
train_copy['Title'] = train_copy['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

rare_titles = ['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
train_copy['Title'] = train_copy['Title'].replace(rare_titles, 'Rare')

# Fixing common French variations
train_copy['Title'] = train_copy['Title'].replace('Mlle', 'Miss')
train_copy['Title'] = train_copy['Title'].replace('Ms', 'Miss')
train_copy['Title'] = train_copy['Title'].replace('Mme', 'Mrs')

train_copy['Title'].value_counts()

In [ ]:
train_copy['Sex'] = train_copy['Sex'].map({'male': 0, 'female': 1})
train_copy = pd.get_dummies(train_copy, columns=['Embarked', 'Title'], prefix=['Emb','Title']) 


In [ ]:
train_copy.tail()

### Exercise 4: Outlier Detection and Handling

Goal: Detect and cap or transform outliers in columns like Fare and Age.

1. Visualize distributions using boxplots or histograms to identify potential outliers.
2. Use IQR or Z-score methods to detect them.
3. Handle outliers with:
*   Quantile capping (e.g. 0.98)
*   Log transformation
*   Row removal
4. Compare the dataset before and after treatment.

📌 Note: Small differences between 0.98 and 0.99 quantiles are normal when extreme values are rare or far apart. Use df.quantile() to explore and choose thresholds empirically, backed by visualization.



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Create a boxplot for Fare
plt.figure(figsize=(10, 5))
sns.boxplot(x=train_copy['Fare'], color='skyblue')
plt.title('Boxplot of Fare (Identifying Outliers)')
plt.show()

In [ ]:
# Create a histogram for Age
plt.figure(figsize=(10, 5))
sns.histplot(train_copy['Age'], kde=True, color='green')
plt.title('Distribution of Passenger Age')
plt.show()

In [ ]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile)
Q1 = train_copy['Fare'].quantile(0.25)
Q3 = train_copy['Fare'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify the outlier rows
outliers_iqr = train_copy[(train_copy['Fare'] < lower_bound) | (train_copy['Fare'] > upper_bound)]

print(f"IQR Upper Bound for Fare: {upper_bound}")
print(f"Number of outliers detected: {len(outliers_iqr)}")

In [ ]:
from scipy import stats
import numpy as np

In [ ]:
# Z-score method for Age (Threshold of 3)
z_age = np.abs(stats.zscore(train_copy['Age']))

print(f"Number of Z-score outliers: {(z_age > 3).sum()}")

In [ ]:
# Cap Age at the 98th percentile
age_98 = train_copy['Age'].quantile(0.98)
train_copy['Age_Capped'] = train_copy['Age'].clip(upper=age_98)

print(f"98th Percentile Age: {age_98}")

In [ ]:
train_copy['Fare_Log'] = np.log1p(train_copy['Fare'])

### Exercise 5: Data Standardization and Normalization

Goal: Scale numerical features to prepare for modeling.

*    Use StandardScaler (mean = 0, std = 1) for normally distributed features.
*    Use MinMaxScaler (range [0, 1]) for features that are skewed or bounded.

📌 Important: Perform this step after outlier treatment to avoid distortion caused by extreme values.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
# 1. Initialize the scalers
std_scaler = StandardScaler()
min_max_scaler = MinMaxScaler()

# 2. Scale Age_Capped (StandardScaler)
# We use [[ ]] because the scaler expects a 2D array (a table, not a list)
train_copy['Age_Scaled'] = std_scaler.fit_transform(train_copy[['Age_Capped']])

# 3. Scale Fare_Log (MinMaxScaler)
train_copy['Fare_Scaled'] = min_max_scaler.fit_transform(train_copy[['Fare_Log']])

# 4. Drop the intermediate columns to keep the final dataset clean
train_final = train_copy.drop(columns=['Age_Capped', 'Fare_Log'])

train_final[['Age_Scaled', 'Fare_Scaled']].describe()

###  Exercise 6: Feature Encoding

Goal: Finalize categorical variable encoding.

1. Identify remaining categorical columns (e.g. Sex, Embarked, Title).
2. Apply:

*   One-Hot Encoding for nominal variables.
*   Label Encoding if any ordinal variables remain.
3. Merge encoded columns back into the main dataset.

📌 Reminder: Encoding comes after handling missing values and outliers, but before scaling (if applicable).

In [ ]:
# Identify and drop columns that cannot be encoded or aren't needed
cols_to_drop = ['Name', 'Ticket', 'PassengerId']
train_final = train_copy.drop(columns=[c for c in cols_to_drop if c in train_copy.columns])

# Final Check: Are there any 'object' columns left?
print(train_final.select_dtypes(include=['object']).columns)

In [ ]:
non_numeric = train_final.select_dtypes(exclude=['number', 'bool']).columns
if len(non_numeric) == 0:
    print("Success! Your dataset is 100% numerical.")
else:
    print(f"Warning: These columns still need encoding: {list(non_numeric)}")

### Exercise 7: Data Transformation for Age Feature

Goal: Create and encode age groups.

1. Use pd.cut() to create bins for life stages (e.g. child, teen, adult, senior).
2. Apply one-hot encoding using pd.get_dummies().

📌 Example: You might define bins like [0, 12, 18, 60, 100] and label them accordingly.

In [ ]:
# Define the age ranges and their names
# 0-12 (Child), 12-18 (Teen), 18-60 (Adult), 60+ (Senior)
bins = [0, 12, 18, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'Senior']

# Use your original 'Age' or 'Age_Capped' column
train_final['AgeGroup'] = pd.cut(train_final['Age_Capped'], bins=bins, labels=labels)

In [ ]:
# Create dummy variables for the new AgeGroup column
train_final = pd.get_dummies(train_final, columns=['AgeGroup'], prefix='Age')

# Show the results
train_final.filter(like='Age_').head()